In [1]:
import os
import sys
os.getcwd()

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0,PROJECT_ROOT)

In [2]:
from backend.features import preprocess_image
from backend.model import load_models, get_models, predict_z

import numpy as np

import tensorflow as tf
import numpy as np


In [ ]:
def predict_z_mdn(predictions):
    """
    mdn output: (6000, 15) — 6000 spatial positions, 5 gaussians × 3 params
    Strategy: mean-pool across spatial dim first, then decode MDN
    """
    def decode_mdn_head(raw):
        # raw: (6000, 15)

        # Step 1 — collapse spatial dimension → (1, 15)
        raw_pooled = tf.reduce_mean(raw, axis=0, keepdims=True)

        # Step 2 — slice the 5 gaussian components
        n = raw_pooled.shape[-1] // 3      # = 5

        pi_logits = raw_pooled[:, :n]      # (1, 5)
        mu        = raw_pooled[:, n:2*n]   # (1, 5)
        # log_sigma = raw_pooled[:, 2*n:]  # (1, 5) — available if needed

        # Step 3 — softmax logits → mixture weights
        pi = tf.nn.softmax(pi_logits, axis=-1)

        # Step 4 — weighted mean redshift
        z = tf.reduce_sum(pi * mu, axis=-1)   # (1,)
        return z, pi

    z1, pi1 = decode_mdn_head(predictions['mdn1_out'])
    z2, pi2 = decode_mdn_head(predictions['mdn2_out'])

    # Confidence-weighted combination of both heads
    confidence1 = tf.reduce_max(pi1, axis=-1)
    confidence2 = tf.reduce_max(pi2, axis=-1)
    total = confidence1 + confidence2 + 1e-8

    z_hat = (confidence1 * z1 + confidence2 * z2) / total

    return tf.cast(z_hat, tf.float64)

In [4]:
# Preprocess Image
np_vals = np.load("data_sample_v4.5_images_0588000_0594000.npy")
loaded_img = preprocess_image(np_vals)


In [ ]:
def explain(preprocess_img: list):
    # 1. Load model and input
    baseline, image = get_models()
    # 2. Identify the target layer (last conv layer)
    # Find the last Conv2D layer in the model
    target_layer_name = None
    for layer in image.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            target_layer_name = layer.name

    if not target_layer_name:
        raise ValueError("No Conv2D layer found in the model.")

    target_layer = image.get_layer(target_layer_name)

    # 3. Create an intermediate model to get activations
    # This model takes the same input but outputs the last conv layer's features
    grad_model = tf.keras.Model(
        inputs=[image.input],
        outputs=[target_layer.output, image.output]
    )

    # 4. Compute gradients using GradientTape
    with tf.GradientTape() as tape:
        # Watch the input tensor if it's not already trainable/variable
        # Usually, we watch the intermediate layer output or the input.
        # For Grad-CAM, we need gradients of the output w.r.t the conv layer features.

        conv_output, mdn_params = grad_model(preprocess_img)

    pi1, mu1, sigma1 = mdn_params[0]
    pi2, mu2, sigma2 = mdn_params[1]
    loss = tf.reduce_sum(pi1 * mu1, axis=-1) + tf.reduce_sum(pi2 * mu2, axis=-1)


    # Calculate gradients of the output w.r.t the convolutional feature map
    grads = tape.gradient(loss, conv_output)

    # 5. Apply Grad-CAM formula
    # Pool gradients over spatial dimensions (H, W) to get alpha_k for each channel
    # conv_output shape: (1, H, W, C)
    # grads shape:       (1, H, W, C)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # Shape: (C,)

    # Weight the feature maps by the gradients
    # Expand dims to match broadcasting: (1, 1, 1, C) * (1, H, W, C)
    cam = conv_output @ pooled_grads[..., tf.newaxis]
    cam = tf.nn.relu(cam)  # ReLU to keep only positive contributions

    # Normalize heatmap to
    cam = tf.keras.activations.softmax(cam, axis=-1) # Optional: softmax per pixel? No, usually global norm.

    # Better normalization: min-max across the whole map
    cam_min = tf.reduce_min(cam)
    cam_max = tf.reduce_max(cam)
    if cam_max > cam_min:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        cam = tf.zeros_like(cam)

    # Convert to numpy
    cam_np = cam.numpy()

    # 6. Upsample to original input size
    #H, W = tensor.shape, tensor.shape
    H, W = 24, 24

    # Resize from (H_cam, W_cam) to (H, W)
    # tf.image.resize expects (H, W, C)
    cam_resized = tf.image.resize(
        cam_np,
        size=[H, W],
        method=tf.image.ResizeMethod.BILINEAR
    ).numpy()[:, :, :, 0]

    predictions = predict_z_mdn(mdn_params)[0]

    return {
        "redshift": predictions,
        "heatmap": cam_resized
    }


In [14]:
explain(preprocess_img=loaded_img)

/Users/cathyzhou/.pyenv/versions/3.12.9/envs/upgrade_macro/lib/python3.12/site-packages/keras/src/models/functional.py:258: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_5ch']
Received: inputs=Tensor(shape=(6000, 24, 24, 5))
  warnings.warn(msg)


KeyError: 0

In [10]:
float(explain(preprocess_img=loaded_img)['redshift'])

/Users/cathyzhou/.pyenv/versions/3.12.9/envs/upgrade_macro/lib/python3.12/site-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (6000, 6, 6, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


0.16060328483581543